In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# List top‑level of your Drive
!ls "/content/drive/My Drive/AI"

cyclegan_sequence_runs	inference_outputs


# Imports

In [3]:
import os
import glob
import random
from pathlib import Path
from tqdm import tqdm

import numpy as np
from PIL import Image, ImageOps
import imageio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# .aps reader

In [5]:
def read_header(infile):
    h = dict()
    fid = open(infile, 'r+b')
    h['filename'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 20))
    h['parent_filename'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 20))
    h['comments1'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 80))
    h['comments2'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 80))
    h['energy_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['config_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['file_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['trans_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['scan_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['data_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['date_modified'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 16))
    h['frequency'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['mat_velocity'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['num_pts'] = np.fromfile(fid, dtype = np.int32, count = 1)
    h['num_polarization_channels'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['spare00'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['adc_min_voltage'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['adc_max_voltage'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['band_width'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['spare01'] = np.fromfile(fid, dtype = np.int16, count = 5)
    h['polarization_type'] = np.fromfile(fid, dtype = np.int16, count = 4)
    h['record_header_size'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['word_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['word_precision'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['min_data_value'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['max_data_value'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['avg_data_value'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['data_scale_factor'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['data_units'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['surf_removal'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['edge_weighting'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['x_units'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['y_units'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['z_units'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['t_units'] = np.fromfile(fid, dtype = np.uint16, count = 1)
    h['spare02'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['x_return_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_return_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_return_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['scan_orientation'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['scan_direction'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['data_storage_order'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['scanner_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['x_inc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_inc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_inc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['t_inc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['num_x_pts'] = np.fromfile(fid, dtype = np.int32, count = 1)
    h['num_y_pts'] = np.fromfile(fid, dtype = np.int32, count = 1)
    h['num_z_pts'] = np.fromfile(fid, dtype = np.int32, count = 1)
    h['num_t_pts'] = np.fromfile(fid, dtype = np.int32, count = 1)
    h['x_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_speed'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['x_acc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_acc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_acc'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['x_motor_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_motor_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_motor_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['x_encoder_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_encoder_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_encoder_res'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['date_processed'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 8))
    h['time_processed'] = b''.join(np.fromfile(fid, dtype = 'S1', count = 8))
    h['depth_recon'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['x_max_travel'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_max_travel'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['elevation_offset_angle'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['roll_offset_angle'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_max_travel'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['azimuth_offset_angle'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['adc_type'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['spare06'] = np.fromfile(fid, dtype = np.int16, count = 1)
    h['scanner_radius'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['x_offset'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['y_offset'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['z_offset'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['t_delay'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['range_gate_start'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['range_gate_end'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['ahis_software_version'] = np.fromfile(fid, dtype = np.float32, count = 1)
    h['spare_end'] = np.fromfile(fid, dtype = np.float32, count = 5)
    fid.close()
    return h

def read_data(infile, scale=True):
    extension = os.path.splitext(infile)[1].lower()
    h = read_header(infile)
    nx = int(h['num_x_pts'])
    ny = int(h['num_y_pts'])
    nt = int(h['num_t_pts'])
    fid = open(infile, 'rb')
    fid.seek(512) #skip header
    if extension == '.aps' or extension == '.a3daps':
        if(int(h['word_type'])==7): #float32
            data = np.fromfile(fid, dtype = np.float32, count = nx * ny * nt)
        elif(int(h['word_type'])==4): #uint16
            data = np.fromfile(fid, dtype = np.uint16, count = nx * ny * nt)
        data = data.reshape(nx, ny, nt, order='F').copy()
        if scale:
            data = data * float(h['data_scale_factor'])
        else:
            data = (data, h['data_scale_factor'])
    elif extension == '.a3d':
        if(int(h['word_type'])==7):
            data = np.fromfile(fid, dtype = np.float32, count = nx * ny * nt)
        elif(int(h['word_type'])==4):
            data = np.fromfile(fid, dtype = np.uint16, count = nx * ny * nt)
        data = data * float(h['data_scale_factor'])
        data = data.reshape(nx, nt, ny, order='F').copy()
    elif extension == '.ahi':
        data = np.fromfile(fid, dtype = np.float32, count = 2* nx * ny * nt)
        data = data.reshape(2, ny, nx, nt, order='F').copy()
        real = data[0,:,:,:].copy()
        imag = data[1,:,:,:].copy()
    fid.close()
    if extension != '.ahi':
        return data
    else:
        return real, imag

def get_x_views(filename, x=16):
    data = read_data(filename)
    views = data.shape[2]
    return [np.flipud(data[:, :, i].transpose()) for i in range(0, views, max(1, views // x))]

def name_to_array(filepath):
    arr_list = get_x_views(filepath, x=16)
    arr = np.array(arr_list)  # (n_views,H,W)
    return arr

# resize+pad utils and stack-prep

In [6]:
def resize_and_pad_pil(pil_img, target_size, pad_value=0):
    target_h, target_w = target_size
    orig_w, orig_h = pil_img.size  # width, height
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(round(orig_w * scale))
    new_h = int(round(orig_h * scale))
    img_resized = pil_img.resize((new_w, new_h), Image.BILINEAR)
    pad_left = (target_w - new_w) // 2
    pad_top  = (target_h - new_h) // 2
    pad_right = target_w - new_w - pad_left
    pad_bottom = target_h - new_h - pad_top
    img_padded = ImageOps.expand(img_resized, border=(pad_left, pad_top, pad_right, pad_bottom), fill=pad_value)
    return img_padded

def prepare_stack_tensor_preserve_aspect(stack_np, target_size):
    n_frames = stack_np.shape[0]
    if n_frames < 16:
        pads = [stack_np[-1]] * (16 - n_frames)
        stack_np = np.concatenate([stack_np, np.stack(pads, axis=0)], axis=0)
        n_frames = 16
    stack_np = stack_np[:16]

    mn = float(np.min(stack_np))
    mx = float(np.max(stack_np))
    if mx - mn < 1e-8:
        scaled_stack = np.zeros_like(stack_np, dtype=np.uint8)
    else:
        scaled_stack = ((stack_np - mn) / (mx - mn) * 255.0).clip(0,255).astype(np.uint8)

    channel_tensors = []
    for i in range(16):
        pil = Image.fromarray(scaled_stack[i])  # 'L'
        pil = resize_and_pad_pil(pil, target_size, pad_value=0)
        t = TF.to_tensor(pil)
        channel_tensors.append(t)
    tensor = torch.cat(channel_tensors, dim=0)
    tensor = tensor * 2.0 - 1.0
    return tensor, (mn, mx)


# Sequence-wise Dataset -> returns full 16-channel tensors

In [7]:
class SequenceStackDataset(Dataset):
    def __init__(self, npy_dir, aps_dir, target_size=(256,256), max_samples=None):
        self.target_size = target_size
        self.npy_paths = sorted(glob.glob(os.path.join(npy_dir, "*.npy")))
        self.aps_paths = sorted(glob.glob(os.path.join(aps_dir, "*.aps")))

        self.npy_list = list(self.npy_paths)
        self.aps_list = list(self.aps_paths)

        if max_samples:
            self.npy_list = random.sample(self.npy_list, min(max_samples, len(self.npy_list)))
            self.aps_list = random.sample(self.aps_list, min(max_samples, len(self.aps_list)))

        self._cache = {'npy_path': None, 'npy_arr': None, 'aps_path': None, 'aps_arr': None}

    def __len__(self):
        return max(len(self.npy_list), len(self.aps_list))

    def _load_npy_stack(self, path):
        arr = np.load(path)
        arr = np.squeeze(arr)
        if arr.ndim == 3 and arr.shape[0] != 16 and arr.shape[1] == 16:
            arr = np.moveaxis(arr, 1, 0)
        return arr.astype(np.float32)

    def _load_aps_stack(self, path):
        arr = name_to_array(path)
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        a_path = self.npy_list[idx % len(self.npy_list)]
        b_path = random.choice(self.aps_list)

        if self._cache['npy_path'] == a_path and self._cache['npy_arr'] is not None:
            a_stack = self._cache['npy_arr']
        else:
            a_stack = self._load_npy_stack(a_path)
            self._cache['npy_path'] = a_path
            self._cache['npy_arr'] = a_stack

        if self._cache['aps_path'] == b_path and self._cache['aps_arr'] is not None:
            b_stack = self._cache['aps_arr']
        else:
            b_stack = self._load_aps_stack(b_path)
            self._cache['aps_path'] = b_path
            self._cache['aps_arr'] = b_stack

        tensor_a, stats_a = prepare_stack_tensor_preserve_aspect(a_stack, self.target_size)
        tensor_b, stats_b = prepare_stack_tensor_preserve_aspect(b_stack, self.target_size)
class SequenceStackDataset(Dataset):
    def __init__(self, npy_dir, aps_dir, target_size=(256,256), max_samples=None):
        self.target_size = target_size
        self.npy_paths = sorted(glob.glob(os.path.join(npy_dir, "*.npy")))
        self.aps_paths = sorted(glob.glob(os.path.join(aps_dir, "*.aps")))

        self.npy_list = list(self.npy_paths)
        self.aps_list = list(self.aps_paths)

        if max_samples:
            self.npy_list = random.sample(self.npy_list, min(max_samples, len(self.npy_list)))
            self.aps_list = random.sample(self.aps_list, min(max_samples, len(self.aps_list)))

        self._cache = {'npy_path': None, 'npy_arr': None, 'aps_path': None, 'aps_arr': None}

    def __len__(self):
        return max(len(self.npy_list), len(self.aps_list))

    def _load_npy_stack(self, path):
        arr = np.load(path)
        arr = np.squeeze(arr)
        if arr.ndim == 3 and arr.shape[0] != 16 and arr.shape[1] == 16:
            arr = np.moveaxis(arr, 1, 0)
        return arr.astype(np.float32)

    def _load_aps_stack(self, path):
        arr = name_to_array(path)
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        a_path = self.npy_list[idx % len(self.npy_list)]
        b_path = random.choice(self.aps_list)


        if self._cache['npy_path'] == a_path and self._cache['npy_arr'] is not None:
            a_stack = self._cache['npy_arr']
        else:
            a_stack = self._load_npy_stack(a_path)
            self._cache['npy_path'] = a_path
            self._cache['npy_arr'] = a_stack

        if self._cache['aps_path'] == b_path and self._cache['aps_arr'] is not None:
            b_stack = self._cache['aps_arr']
        else:
            b_stack = self._load_aps_stack(b_path)
            self._cache['aps_path'] = b_path
            self._cache['aps_arr'] = b_stack

        tensor_a, stats_a = prepare_stack_tensor_preserve_aspect(a_stack, self.target_size)
        tensor_b, stats_b = prepare_stack_tensor_preserve_aspect(b_stack, self.target_size)

        return {'A': tensor_a, 'B': tensor_b, 'A_path': a_path, 'B_path': b_path, 'A_stats': stats_a, 'B_stats': stats_b}

        return {'A': tensor_a, 'B': tensor_b, 'A_path': a_path, 'B_path': b_path, 'A_stats': stats_a, 'B_stats': stats_b}


# Models

In [8]:
class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.InstanceNorm2d, use_dropout=False):
        super().__init__()
        block = []
        block += [nn.ReflectionPad2d(1),
                  nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
                  norm_layer(dim),
                  nn.ReLU(True)]
        if use_dropout:
            block += [nn.Dropout(0.5)]
        block += [nn.ReflectionPad2d(1),
                  nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
                  norm_layer(dim)]
        self.block = nn.Sequential(*block)

    def forward(self, x):
        return x + self.block(x)

class ResnetGeneratorMultiChannel(nn.Module):
    def __init__(self, input_nc=16, output_nc=16, ngf=64, n_blocks=9, norm_layer=nn.InstanceNorm2d):
        super().__init__()
        model = []
        model += [nn.ReflectionPad2d(3),
                  nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=True),
                  norm_layer(ngf),
                  nn.ReLU(True)]

        n_downsampling = 2
        mult = 1
        for i in range(n_downsampling):
            mult_prev = mult
            mult = mult * 2
            model += [nn.Conv2d(ngf * mult_prev, ngf * mult, kernel_size=3, stride=2, padding=1, bias=True),
                      norm_layer(ngf * mult),
                      nn.ReLU(True)]
        for i in range(n_blocks):
            model += [ResnetBlock(ngf * mult, norm_layer=norm_layer)]
        for i in range(n_downsampling):
            mult_prev = mult
            mult = mult // 2
            model += [nn.ConvTranspose2d(ngf * mult_prev, ngf * mult, kernel_size=3, stride=2,
                                         padding=1, output_padding=1, bias=True),
                      norm_layer(ngf * mult),
                      nn.ReLU(True)]
        model += [nn.ReflectionPad2d(3),
                  nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0),
                  nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

class NLayerDiscriminatorMultiChannel(nn.Module):
    def __init__(self, input_nc=16, ndf=64, n_layers=3):
        super().__init__()
        kw = 4
        padw = 1
        sequence = [
            nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw),
            nn.LeakyReLU(0.2, True)
        ]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            sequence += [
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=2, padding=padw, bias=False),
                nn.InstanceNorm2d(ndf * nf_mult),
                nn.LeakyReLU(0.2, True)
            ]
        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        sequence += [
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=1, padding=padw, bias=False),
            nn.InstanceNorm2d(ndf * nf_mult),
            nn.LeakyReLU(0.2, True)
        ]
        sequence += [nn.Conv2d(ndf * nf_mult, 1, kernel_size=kw, stride=1, padding=padw)]
        self.model = nn.Sequential(*sequence)

    def forward(self, x):
        return self.model(x)


# utils

In [9]:
def init_weights(net, init_type='normal', init_gain=0.02):
    def init_fn(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            if init_type == 'normal':
                nn.init.normal_(m.weight.data, 0.0, init_gain)
            elif init_type == 'xavier':
                nn.init.xavier_normal_(m.weight.data, gain=init_gain)
            elif init_type == 'kaiming':
                nn.init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif classname.find('BatchNorm2d') != -1 or classname.find('InstanceNorm') != -1:
            if hasattr(m, 'weight') and m.weight is not None:
                nn.init.normal_(m.weight.data, 1.0, init_gain)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
    net.apply(init_fn)

def stack_tensor_to_uint8_frames(tensor_stack):
    if tensor_stack.dim() == 4:
        tensor_stack = tensor_stack[0]  # take first sample if batch
    C, H, W = tensor_stack.shape
    frames = []
    for c in range(C):
        t = tensor_stack[c, :, :].detach().cpu()
        img = (t + 1.0) / 2.0
        arr = (img.numpy() * 255.0).clip(0,255).astype(np.uint8)
        frames.append(arr)
    return frames


# Sequence-wise trainer

In [10]:
class CycleGANSequenceTrainer:
    def __init__(self, device='cpu', input_nc=16, output_nc=16, ngf=64, ndf=64, lr=2e-4,
                 lambda_cycle=10.0, lambda_id=0.5, use_identity=True):
        self.device = device
        self.G_A = ResnetGeneratorMultiChannel(input_nc=input_nc, output_nc=output_nc, ngf=ngf).to(device)
        self.G_B = ResnetGeneratorMultiChannel(input_nc=output_nc, output_nc=input_nc, ngf=ngf).to(device)
        self.D_A = NLayerDiscriminatorMultiChannel(input_nc=output_nc, ndf=ndf).to(device)
        self.D_B = NLayerDiscriminatorMultiChannel(input_nc=input_nc, ndf=ndf).to(device)

        init_weights(self.G_A); init_weights(self.G_B); init_weights(self.D_A); init_weights(self.D_B)

        self.criterion_GAN = nn.MSELoss().to(device)
        self.criterion_cycle = nn.L1Loss().to(device)
        self.criterion_identity = nn.L1Loss().to(device)

        self.optimizer_G = torch.optim.Adam(list(self.G_A.parameters()) + list(self.G_B.parameters()), lr=lr, betas=(0.5, 0.999))
        self.optimizer_D = torch.optim.Adam(list(self.D_A.parameters()) + list(self.D_B.parameters()), lr=lr, betas=(0.5, 0.999))

        self.lambda_cycle = lambda_cycle
        self.lambda_id = lambda_id if use_identity else 0.0

    def set_requires_grad(self, nets, requires_grad=False):
        if not isinstance(nets, list):
            nets = [nets]
        for net in nets:
            for p in net.parameters():
                p.requires_grad = requires_grad

    def train_step(self, real_A, real_B):
        with torch.no_grad():
            dummy = self.D_A(real_B)
        real_label = torch.ones_like(dummy, device=self.device)
        fake_label = torch.zeros_like(dummy, device=self.device)

        # Generators
        self.set_requires_grad([self.D_A, self.D_B], False)
        self.optimizer_G.zero_grad()

        idt_A = self.G_B(real_A)
        idt_B = self.G_A(real_B)
        loss_idt = (self.criterion_identity(idt_A, real_A) + self.criterion_identity(idt_B, real_B)) * self.lambda_id

        fake_B = self.G_A(real_A)
        pred_fake_B = self.D_A(fake_B)
        loss_GAN_A = self.criterion_GAN(pred_fake_B, real_label)

        fake_A = self.G_B(real_B)
        pred_fake_A = self.D_B(fake_A)
        loss_GAN_B = self.criterion_GAN(pred_fake_A, real_label)

        rec_A = self.G_B(fake_B)
        rec_B = self.G_A(fake_A)
        loss_cycle = self.criterion_cycle(rec_A, real_A) * self.lambda_cycle + \
                     self.criterion_cycle(rec_B, real_B) * self.lambda_cycle

        loss_G = loss_GAN_A + loss_GAN_B + loss_cycle + loss_idt
        loss_G.backward()
        self.optimizer_G.step()

        # Discriminators
        self.set_requires_grad([self.D_A, self.D_B], True)
        self.optimizer_D.zero_grad()

        pred_real_B = self.D_A(real_B)
        loss_D_A_real = self.criterion_GAN(pred_real_B, real_label)
        pred_fake_B = self.D_A(fake_B.detach())
        loss_D_A_fake = self.criterion_GAN(pred_fake_B, fake_label)
        loss_D_A = (loss_D_A_real + loss_D_A_fake) * 0.5

        pred_real_A = self.D_B(real_A)
        loss_D_B_real = self.criterion_GAN(pred_real_A, real_label)
        pred_fake_A = self.D_B(fake_A.detach())
        loss_D_B_fake = self.criterion_GAN(pred_fake_A, fake_label)
        loss_D_B = (loss_D_B_real + loss_D_B_fake) * 0.5

        loss_D = loss_D_A + loss_D_B
        loss_D.backward()
        self.optimizer_D.step()

        return {
            'loss_G': loss_G.item(),
            'loss_GAN_A': loss_GAN_A.item(), 'loss_GAN_B': loss_GAN_B.item(),
            'loss_cycle': loss_cycle.item(), 'loss_idt': loss_idt.item(),
            'loss_D': loss_D.item()
        }

    def translate_A_to_B(self, real_A):
        self.G_A.eval()
        with torch.no_grad():
            fake_B = self.G_A(real_A.to(self.device))
        self.G_A.train()
        return fake_B


# Training runner

In [ ]:
npy_dir = "/content/drive/My Drive/project/AI/Sample_MMW_Dataset/Sightence_Sample_Data_Unsupervised_temp"
aps_dir = "/content/drive/My Drive/project/AI/Sample_MMW_Dataset/Kaggle_Sample_Data_Unsupervised"
target_size = (256, 256)
batch_size = 1
num_epochs = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
save_dir = "/content/drive/My Drive/AI/cyclegan_sequence_runs"
os.makedirs(save_dir, exist_ok=True)


dataset = SequenceStackDataset(npy_dir=npy_dir, aps_dir=aps_dir, target_size=target_size)


if len(dataset) == 0:
    raise RuntimeError(f"Dataset appears empty. Check npy_dir='{npy_dir}' and aps_dir='{aps_dir}' for files. len(dataset)={len(dataset)}")


num_workers = 4
if os.name == 'nt':
    num_workers = 0

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)


print(f"Dataset length: {len(dataset)} samples. batch_size={batch_size}, num_workers={num_workers}")

trainer = CycleGANSequenceTrainer(device=device, input_nc=16, output_nc=16, ngf=64, ndf=64, lr=2e-4)


niter = num_epochs // 2
niter_decay = num_epochs - niter
print(f"LR schedule: {niter} epochs constant, {niter_decay} epochs linear decay (total {num_epochs})")

from torch.optim.lr_scheduler import LambdaLR

def lambda_rule(epoch):
    if epoch < niter:
        return 1.0
    else:
        return max(0.0, 1.0 - float(epoch - niter) / float(max(1, niter_decay)))

scheduler_G = LambdaLR(trainer.optimizer_G, lr_lambda=lambda_rule)
scheduler_D = LambdaLR(trainer.optimizer_D, lr_lambda=lambda_rule)
# -----------------------

global_step = 0
for epoch in range(1, num_epochs + 1):
    pbar = tqdm(enumerate(loader), total=len(loader))
    epoch_losses = []
    for i, batch in pbar:
        real_A = batch['A'].to(device)
        real_B = batch['B'].to(device)
        losses = trainer.train_step(real_A, real_B)
        epoch_losses.append(losses['loss_G'])
        pbar.set_description(f"epoch{epoch} lossG {losses['loss_G']:.4f} lossD {losses['loss_D']:.4f}")
        global_step += 1

    # Save checkpoint
    ckpt = {
        'G_A': trainer.G_A.state_dict(),
        'G_B': trainer.G_B.state_dict(),
        'D_A': trainer.D_A.state_dict(),
        'D_B': trainer.D_B.state_dict(),
        'epoch': epoch
    }
    torch.save(ckpt, os.path.join(save_dir, f"ckpt_epoch_{epoch}.pth"))


    try:
        sample = next(iter(loader))
        real_A = sample['A'][:1].to(device)   # single stack
        fake_B = trainer.translate_A_to_B(real_A)   # (1,16,H,W)
        frames_uint8 = stack_tensor_to_uint8_frames(fake_B)  # list length 16
        np.save(os.path.join(save_dir, f"sample_fakeB_epoch{epoch}.npy"), np.stack(frames_uint8, axis=0))
        imageio.mimsave(os.path.join(save_dir, f"sample_fakeB_epoch{epoch}.gif"), frames_uint8, fps=4)
    except Exception as e:
        print(f"Warning: failed to create/save sample for epoch {epoch}: {e}")

    scheduler_G.step()
    scheduler_D.step()

    current_lr_G = trainer.optimizer_G.param_groups[0]['lr']
    current_lr_D = trainer.optimizer_D.param_groups[0]['lr']

    print(f"Epoch {epoch} done. avg loss_G {np.mean(epoch_losses):.4f} | lr_G={current_lr_G:.6e} lr_D={current_lr_D:.6e}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataset length: 50 samples. batch_size=1, num_workers=4
LR schedule: 50 epochs constant, 50 epochs linear decay (total 100)


  0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ny = int(h['num_y_pts'])
/tmp/ipython-input-2316629062.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nt = int(h['num_t_pts'])
/tmp/ipython-input-2316629062.py:100: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar 

Epoch 1 done. avg loss_G 7.1816 | lr_G=2.000000e-04 lr_D=2.000000e-04


  0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ny = int(h['num_y_pts'])
/tmp/ipython-input-2316629062.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nt = int(h['num_t_pts'])
/tmp/ipython-input-2316629062.py:100: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar 

...

Epoch 98 done. avg loss_G 0.9155 | lr_G=8.000000e-06 lr_D=8.000000e-06


  0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ny = int(h['num_y_pts'])
/tmp/ipython-input-2316629062.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar i

Epoch 99 done. avg loss_G 0.9255 | lr_G=4.000000e-06 lr_D=4.000000e-06


  0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ny = int(h['num_y_pts'])
/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  nx = int(h['num_x_pts'])
/tmp/ipython-input-2316629062.py:94: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar i

Epoch 100 done. avg loss_G 0.9138 | lr_G=0.000000e+00 lr_D=0.000000e+00


#  Integrated Test / Inference

In [13]:
target_size = (256, 256)
batch_size = 1
num_epochs = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt_path = "/content/drive/My Drive/AI/cyclegan_sequence_runs/ckpt_epoch_100.pth"
test_npy_path = "/content/drive/My Drive/project/AI/Sample_MMW_Dataset/Sightence_Sample_Data_Unsupervised_temp/AI_2023_10_11_16_16_06_1459_f_enhanced.npy"
out_dir = "/content/drive/My Drive/AI/inference_outputs"
os.makedirs(out_dir, exist_ok=True)
# -----------------------------------------------------------------------
trainer = CycleGANSequenceTrainer(device=device, input_nc=16, output_nc=16, ngf=64, ndf=64, lr=2e-4)

ckpt = torch.load(ckpt_path, map_location=device)
trainer.G_A.load_state_dict(ckpt['G_A'])
trainer.G_B.load_state_dict(ckpt['G_B'])
trainer.G_A.to(device)
trainer.G_A.eval()

stack = np.load(test_npy_path)
stack = np.squeeze(stack)


tensor_stack, stats = prepare_stack_tensor_preserve_aspect(stack, target_size)  # tensor shape (16,H,W) with values in [-1,1]
mn, mx = stats


original_frames_uint8 = stack_tensor_to_uint8_frames(tensor_stack)   # CHANGED: list of 16 uint8 arrays (H,W)
np.save(os.path.join(out_dir, "original_uint8_stack.npy"), np.stack(original_frames_uint8, axis=0))  # CHANGED
imageio.mimsave(os.path.join(out_dir, "original_visual.gif"), original_frames_uint8, fps=4)  # CHANGED


with torch.no_grad():
    input_tensor = tensor_stack.unsqueeze(0).to(device)
    fake = trainer.G_A(input_tensor)
fake_cpu = fake.squeeze(0).cpu()


frames_uint8 = stack_tensor_to_uint8_frames(fake_cpu)
np.save(os.path.join(out_dir, "translated_uint8_stack.npy"), np.stack(frames_uint8, axis=0))
imageio.mimsave(os.path.join(out_dir, "translated_visual.gif"), frames_uint8, fps=4)


stack_uint8 = np.stack(frames_uint8, axis=0).astype(np.float32)

stack_float_original_range = (stack_uint8 / 255.0) * (mx - mn) + mn
np.save(os.path.join(out_dir, "translated_float_original_range.npy"), stack_float_original_range)

print("Inference done.")
print("Saved:")
print(" - original_uint8_stack.npy")
print(" - original_visual.gif")
print(" - translated_uint8_stack.npy")
print(" - translated_visual.gif")
print(" - translated_float_original_range.npy")
print("All saved in:", out_dir)


Inference done.
Saved:
 - original_uint8_stack.npy
 - original_visual.gif
 - translated_uint8_stack.npy
 - translated_visual.gif
 - translated_float_original_range.npy
All saved in: /content/drive/My Drive/AI/inference_outputs
